# 14 Transformer 内部结构、Attention、RoPE 和 GQA

目标：把面试中常见的结构问题落到形状和公式上：hidden size、head dim、MHA/GQA/MQA、RoPE、attention 显存复杂度、参数量和 KV cache。


## 1. 安装依赖


In [ ]:
from pathlib import Path

base = Path.cwd()
requirements_path = base / "requirements.txt"
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

print("requirements:", requirements_path)
%pip install -r {requirements_path}


## 2. 读取模型配置

很多结构问题不需要先加载完整权重，先看 `config.json` 就能回答大半。


In [ ]:
import math
import os
from pathlib import Path

import torch
from transformers import AutoConfig

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("MODEL_ID =", MODEL_ID)
print("model_type =", getattr(config, "model_type", None))

fields = [
    "hidden_size",
    "num_hidden_layers",
    "num_attention_heads",
    "num_key_value_heads",
    "intermediate_size",
    "vocab_size",
    "max_position_embeddings",
    "rope_theta",
    "sliding_window",
]
for name in fields:
    print(f"{name:24s}", getattr(config, name, None))


## 3. 从配置推导 head_dim、GQA 分组和 KV cache

GQA 的关键是 query heads 多，key/value heads 少。decode 时每个新 token 只追加 K/V，所以 KV cache 大小和 `num_key_value_heads` 直接相关。


In [ ]:
hidden_size = getattr(config, "hidden_size")
num_layers = getattr(config, "num_hidden_layers")
num_heads = getattr(config, "num_attention_heads")
num_kv_heads = getattr(config, "num_key_value_heads", num_heads)
head_dim = hidden_size // num_heads
groups = num_heads // num_kv_heads

print("head_dim:", head_dim)
print("attention mode:", "MHA" if num_kv_heads == num_heads else "GQA/MQA")
print("query heads per kv head:", groups)


def kv_cache_gib(batch_size, seq_len, dtype_bytes=2):
    bytes_count = batch_size * seq_len * num_layers * 2 * num_kv_heads * head_dim * dtype_bytes
    return bytes_count / 1024**3

for seq_len in [1024, 4096, 8192, 32768]:
    print(f"batch=1 seq={seq_len:<6} fp16/bf16 KV cache ~= {kv_cache_gib(1, seq_len):.3f} GiB")


## 4. 手写一个小 attention，观察形状

真实模型会用 FlashAttention 等 fused kernel，但形状关系不变。标准 attention score 是 `[batch, heads, query_len, key_len]`，因此训练或 full attention 的中间矩阵是长度平方级。


In [ ]:
torch.manual_seed(0)

batch_size = 2
num_demo_heads = 4
seq_len = 5
demo_head_dim = 8

q = torch.randn(batch_size, num_demo_heads, seq_len, demo_head_dim)
k = torch.randn(batch_size, num_demo_heads, seq_len, demo_head_dim)
v = torch.randn(batch_size, num_demo_heads, seq_len, demo_head_dim)

scores = q @ k.transpose(-2, -1) / math.sqrt(demo_head_dim)
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
scores = scores.masked_fill(causal_mask, float("-inf"))
weights = torch.softmax(scores, dim=-1)
context = weights @ v

print("q:", tuple(q.shape))
print("k:", tuple(k.shape))
print("scores:", tuple(scores.shape))
print("weights row sums:", weights[0, 0].sum(dim=-1))
print("context:", tuple(context.shape))


## 5. MHA、GQA、MQA 的形状差异

- MHA：Q/K/V head 数相同，表达力强但 KV cache 大。
- GQA：多个 Q heads 共享一组 K/V heads，兼顾质量和缓存。
- MQA：所有 Q heads 共享一组 K/V heads，缓存最省但质量可能受影响。


In [ ]:
demo_q_heads = 8
for demo_kv_heads in [8, 2, 1]:
    q = torch.randn(1, demo_q_heads, 4, 16)
    k = torch.randn(1, demo_kv_heads, 4, 16)
    v = torch.randn(1, demo_kv_heads, 4, 16)

    repeat = demo_q_heads // demo_kv_heads
    expanded_k = k.repeat_interleave(repeat, dim=1)
    expanded_v = v.repeat_interleave(repeat, dim=1)

    mode = "MHA" if demo_kv_heads == demo_q_heads else ("MQA" if demo_kv_heads == 1 else "GQA")
    print("=" * 80)
    print(mode, "q_heads=", demo_q_heads, "kv_heads=", demo_kv_heads, "repeat=", repeat)
    print("q:", tuple(q.shape), "k before:", tuple(k.shape), "k after:", tuple(expanded_k.shape))


## 6. RoPE 的直觉实验

RoPE 会按位置旋转 q/k 向量。相同位置一起旋转时向量范数不变；不同位置之间的点积会携带相对位置信息。


In [ ]:
def apply_rope(x, position, theta=10000.0):
    x = x.clone()
    dim = x.shape[-1]
    idx = torch.arange(0, dim, 2, dtype=torch.float32)
    inv_freq = 1.0 / (theta ** (idx / dim))
    angles = position * inv_freq
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    even = x[..., 0::2].clone()
    odd = x[..., 1::2].clone()
    x[..., 0::2] = even * cos - odd * sin
    x[..., 1::2] = even * sin + odd * cos
    return x

vec_q = torch.randn(8)
vec_k = torch.randn(8)

q_pos_3 = apply_rope(vec_q, position=3)
k_pos_3 = apply_rope(vec_k, position=3)
k_pos_8 = apply_rope(vec_k, position=8)

print("norm before:", float(vec_q.norm()))
print("norm after :", float(q_pos_3.norm()))
print("dot original:", float(torch.dot(vec_q, vec_k)))
print("dot same position:", float(torch.dot(q_pos_3, k_pos_3)))
print("dot different positions:", float(torch.dot(q_pos_3, k_pos_8)))


## 7. 粗估参数量

这个估算忽略 bias、norm 和少量细节，但足够解释为什么 embedding、attention projection 和 MLP 是参数大头。


In [ ]:
intermediate_size = getattr(config, "intermediate_size")
vocab_size = getattr(config, "vocab_size")

q_proj = hidden_size * hidden_size
kv_proj = 2 * hidden_size * num_kv_heads * head_dim
o_proj = hidden_size * hidden_size
attn_per_layer = q_proj + kv_proj + o_proj
mlp_per_layer = 3 * hidden_size * intermediate_size
embed = vocab_size * hidden_size
rough_total = num_layers * (attn_per_layer + mlp_per_layer) + embed

print(f"q_proj per layer       {q_proj/1e6:10.2f} M")
print(f"k/v proj per layer     {kv_proj/1e6:10.2f} M")
print(f"o_proj per layer       {o_proj/1e6:10.2f} M")
print(f"attention per layer    {attn_per_layer/1e6:10.2f} M")
print(f"MLP per layer          {mlp_per_layer/1e6:10.2f} M")
print(f"embedding              {embed/1e6:10.2f} M")
print(f"rough total            {rough_total/1e9:10.2f} B")


## 面试总结

- `head_dim = hidden_size / num_attention_heads`。
- MHA、GQA、MQA 的核心差异是 K/V heads 数量，直接影响 KV cache 和 decode 显存。
- prefill 阶段通常受大矩阵计算影响，decode 阶段更容易受 KV cache 读写和 batch 调度影响。
- RoPE 不是简单加 position embedding，而是对 q/k 做位置相关旋转。
- attention 中间分数是 `query_len * key_len` 级别，长上下文优化要关注 FlashAttention、paged attention、sliding window、prefix cache。
- 面试回答结构问题时，最好能同时说出 shape、公式和对部署资源的影响。
